In [1]:
# Blocco 1 — Imports e verifica ambiente

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate

# Verifica device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device in uso: {device}")
print(f"Versione PyTorch: {torch.__version__}")

/Users/andrea.pistolato/Desktop/Directory/ML-Data-AI-Portfolio/C3_Fine-Tuning_a_Pre-Trained_Model/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device in uso: mps
Versione PyTorch: 2.10.0


In [2]:
# Blocco 2 — Caricamento e ispezione dataset SST-2

dataset = load_dataset("glue", "sst2")

print(dataset)

Generating test split: 100%|██████████| 1821/1821 [00:00<00:00, 624056.51 examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [3]:
# Blocco 3 — Ispezione dettagliata

print("Colonne:", dataset["train"].column_names)
print("Numero esempi — train:", len(dataset["train"]))
print("Numero esempi — validation:", len(dataset["validation"]))
print("Numero esempi — test:", len(dataset["test"]))

print("\nPrimi 3 esempi dal training set:")
for i in range(3):
    print(dataset["train"][i])

Colonne: ['sentence', 'label', 'idx']
Numero esempi — train: 67349
Numero esempi — validation: 872
Numero esempi — test: 1821

Primi 3 esempi dal training set:
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
{'sentence': 'contains no wit , only labored gags ', 'label': 0, 'idx': 1}
{'sentence': 'that loves its characters and communicates something rather beautiful about human nature ', 'label': 1, 'idx': 2}


In [4]:
# Blocco 4 — Caricamento e ispezione tokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenizziamo una frase di esempio
frase = "This movie was absolutely wonderful!"
output = tokenizer(frase)

print("Token IDs:", output["input_ids"])
print("Attention mask:", output["attention_mask"])
print("\nToken decodificati:")
print(tokenizer.convert_ids_to_tokens(output["input_ids"]))

Token IDs: [101, 2023, 3185, 2001, 7078, 6919, 999, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1]

Token decodificati:
['[CLS]', 'this', 'movie', 'was', 'absolutely', 'wonderful', '!', '[SEP]']


In [5]:
# Blocco 5 — Padding e attention mask su batch

frasi = [
    "good",
    "this film was absolutely wonderful and i loved it"
]

output_batch = tokenizer(
    frasi,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Input IDs:")
print(output_batch["input_ids"])
print("\nAttention mask:")
print(output_batch["attention_mask"])

Input IDs:
tensor([[ 101, 2204,  102,    0,    0,    0,    0,    0,    0,    0,    0],
        [ 101, 2023, 2143, 2001, 7078, 6919, 1998, 1045, 3866, 2009,  102]])

Attention mask:
tensor([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [6]:
# Blocco 6 — Caricamento modello baseline

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model = model.to(device)

print(f"Modello caricato su: {device}")
print(f"Numero parametri totali: {sum(p.numel() for p in model.parameters()):,}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11716.26it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modello caricato su: mps
Numero parametri totali: 66,955,010


In [7]:
# Blocco 7 — Predizioni baseline sul validation set

import numpy as np
from torch.utils.data import DataLoader

# Prepariamo il validation set
def tokenize_batch(batch):
    return tokenizer(
        batch["sentence"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

# Mettiamo il modello in modalità evaluation
model.eval()

all_preds = []
all_labels = []

# Processiamo in batch da 32
validation_data = dataset["validation"]
batch_size = 32

for i in range(0, len(validation_data), batch_size):
    batch = validation_data[i : i + batch_size]
    
    # Tokenizziamo
    inputs = tokenize_batch(batch)
    
    # Spostiamo su MPS
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Forward pass senza gradients
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Estraiamo le predizioni
    logits = outputs.logits
    preds = torch.argmax(logits, dim=-1)
    
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch["label"])

# Calcoliamo accuracy
accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Baseline accuracy sul validation set: {accuracy:.4f} ({accuracy*100:.2f}%)")

Baseline accuracy sul validation set: 0.5344 (53.44%)


In [10]:
# Blocco 8 — Classification report dettagliato

from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report — Baseline:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=["Negative (0)", "Positive (1)"]
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds, labels=[0, 1]))

Classification Report — Baseline:
              precision    recall  f1-score   support

Negative (0)       0.54      0.39      0.45       428
Positive (1)       0.53      0.68      0.60       444

    accuracy                           0.53       872
   macro avg       0.53      0.53      0.52       872
weighted avg       0.53      0.53      0.52       872

Confusion Matrix:
[[165 263]
 [143 301]]


Ambiente verificato ✓
Dataset SST-2 caricato e ispezionato ✓
Tokenizer compreso ✓
Modello baseline valutato ✓
Accuracy baseline: 53.44%